# RAG Evaluation Lab — Bramblewood Gear Co. / FieldWise

**Company:** Bramblewood Gear Co., an outdoor equipment retailer.
**Product:** FieldWise, the internal assistant support agents use while
working a ticket.

Before FieldWise goes in front of real agents, Bramblewood wants to know
whether it's actually accurate. This notebook builds a small internal
evaluation set from FieldWise's own support knowledge base -- not a public
benchmark -- because that's what an eval set should reflect: the questions
FieldWise will really be asked, checked against reference answers a
support-ops lead actually signed off on.

It uses a local Chroma vector store, Groq for generation, and LangSmith for
dataset management, evaluation, and experiment comparison, each with an
offline-safe fallback so the notebook runs the same way with or without
live credentials.

## What this notebook does

1. Builds a small internal support knowledge base and a matching
   question/reference-answer eval set.
2. Builds a Chroma index from that knowledge base.
3. Runs a Groq-powered RAG chain (with an offline fallback).
4. Creates a LangSmith dataset from the eval set (with an offline
   fallback).
5. Evaluates a baseline chain and a query-rewrite chain.
6. Compares results side by side.
7. Computes simple RAGAS-style proxy metrics locally for quick inspection.

## 1) Install packages

In [ ]:
%pip install -qU python-dotenv pandas langsmith langchain langchain-community langchain-text-splitters langchain-groq langchain-chroma chromadb scikit-learn

## 2) Environment variables

Set these in a `.env` file or your shell:

```env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=...
LANGSMITH_PROJECT=fieldwise-rag-eval
GROQ_API_KEY=...
GROQ_MODEL=llama-3.3-70b-versatile
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "fieldwise-rag-eval")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")

print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))
print("LANGSMITH_PROJECT:", LANGSMITH_PROJECT)
print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("GROQ_MODEL:", GROQ_MODEL)

if not LANGSMITH_API_KEY:
    os.environ["LANGSMITH_TRACING"] = "false"

## 3) Build FieldWise's support knowledge base and eval set

Ten short support-knowledge-base articles covering Bramblewood's product
lines and policies, and sixteen question/reference-answer pairs a
support-ops lead would actually sign off on -- this is the internal
equivalent of a public RAG benchmark, built from the company's own content
instead of Wikipedia.

In [ ]:
from langchain_core.documents import Document

support_articles = [
    ("tent_waterproofing", (
        "Tent rainflies use a factory-applied waterproof coating that wears "
        "down with UV exposure. If a rainfly leaks, reapply a DWR (durable "
        "water repellent) treatment once a year and reseal the seams with "
        "seam sealer. If leaking continues after reproofing and the tent is "
        "less than two years old, treat it as a hard-goods warranty claim "
        "rather than a general return."
    )),
    ("tent_models", (
        "The Summit Ridge 2P Tent is a 3-season, freestanding tent with a "
        "silicone-coated rainfly. The Alpine Trail 4P Tent is a 4-season "
        "expedition tent with a HydroShield-coated rainfly and reinforced "
        "poles rated for high wind and snow load."
    )),
    ("tent_poles", (
        "Aluminum tent poles can develop hairline cracks near the ferrules "
        "after repeated setup in high wind. A cracked pole should be "
        "repaired with a pole splint before the next trip; a snapped pole "
        "within the warranty period is replaced at no cost."
    )),
    ("sleeping_bag_care", (
        "The Ember Down sleeping bag loses loft when washed with regular "
        "detergent, which strips the water-resistant DriDown treatment. "
        "Wash only with technical down soap on a gentle cycle and tumble "
        "dry on low heat with a few clean tennis balls to restore loft."
    )),
    ("backpack_warranty", (
        "Basecamp 65L Backpack zipper sliders can stick after heavy use; a "
        "dry silicone lubricant usually frees a stiff slider. A fully "
        "broken slider is covered under the lifetime workmanship warranty "
        "and can be repaired or replaced at no cost."
    )),
    ("stove_troubleshooting", (
        "The TrailFlame Compact Stove's piezo igniter can stop sparking "
        "after repeated use in wet or cold conditions. Any lighter or "
        "match can ignite the burner while the igniter is unavailable. A "
        "piezo igniter that fails within two years is covered under the "
        "standard hard-goods warranty. Fuel canisters from other brands "
        "with a standard threaded valve are compatible, though Bramblewood "
        "only warranties performance with its own TrailFlame canisters."
    )),
    ("footwear_care", (
        "Ridgeline Hiking Boots use a waterproof membrane that typically "
        "needs reproofing after a full season of regular use. An annual "
        "spray-on DWR treatment restores it. Once the outsole wears down, "
        "boots can go through Bramblewood's resole program for a flat fee, "
        "separate from the one-year manufacturing warranty."
    )),
    ("returns_policy", (
        "Unused gear in its original packaging can be returned within 60 "
        "days of purchase for a full refund. Gear that has been used or "
        "worn is not eligible for a general return; it is handled through "
        "a warranty claim instead, if the issue matches a covered defect "
        "type."
    )),
    ("shipping_and_damage", (
        "Orders that arrive damaged should be reported to support with "
        "photos of the damage so a replacement or refund can be "
        "processed. This is handled separately from the standard 60-day "
        "return window and doesn't require the original packaging."
    )),
    ("support_contact", (
        "Bramblewood support is available by chat and email through the "
        "FieldWise-powered support console. Typical response time is "
        "within one business day; urgent warranty and safety issues are "
        "prioritized."
    )),
]

documents = [
    Document(page_content=text, metadata={"title": title, "source": "fieldwise_handbook"})
    for title, text in support_articles
]

qa_examples = [
    {"inputs": {"question": "How do I fix a leaking tent rainfly?"},
     "outputs": {"answer": "Reapply a DWR treatment to the rainfly and reseal the seams with seam sealer; if leaking continues within two years, it's a warranty claim."}},
    {"inputs": {"question": "What's the warranty on the Basecamp backpack?"},
     "outputs": {"answer": "The Basecamp 65L Backpack carries a lifetime warranty against defects in materials and workmanship under normal use."}},
    {"inputs": {"question": "How should I wash my Ember Down sleeping bag?"},
     "outputs": {"answer": "Wash only with technical down soap on a gentle cycle and tumble dry on low heat with clean tennis balls to restore loft; regular detergent strips the DriDown treatment."}},
    {"inputs": {"question": "My TrailFlame stove's igniter won't spark, what should I do?"},
     "outputs": {"answer": "Use a backup lighter or match to ignite the burner; a piezo igniter that fails within two years is covered under the standard hard-goods warranty."}},
    {"inputs": {"question": "Can I return a tent I've already used?"},
     "outputs": {"answer": "No, used gear is not eligible for the standard 60-day return; it can only be handled through a warranty claim if it matches a covered defect."}},
    {"inputs": {"question": "How long is the return window for unused gear?"},
     "outputs": {"answer": "60 days from purchase, for unused gear in its original packaging."}},
    {"inputs": {"question": "How do I stop my hiking boots from leaking?"},
     "outputs": {"answer": "Apply an annual DWR reproofing spray to restore the waterproof membrane."}},
    {"inputs": {"question": "Is boot resoling covered under warranty?"},
     "outputs": {"answer": "No, resoling is a separate paid program available for the life of the boot, separate from the one-year manufacturing warranty."}},
    {"inputs": {"question": "What's the warranty on the TrailFlame stove?"},
     "outputs": {"answer": "2 years against manufacturing defects."}},
    {"inputs": {"question": "My backpack zipper is stuck, what should I try first?"},
     "outputs": {"answer": "Apply a dry silicone lubricant along the zipper teeth before treating it as a defect."}},
    {"inputs": {"question": "Are other brands' fuel canisters compatible with the TrailFlame stove?"},
     "outputs": {"answer": "Yes, canisters with a standard threaded valve are compatible, but Bramblewood only warranties performance with its own TrailFlame canisters."}},
    {"inputs": {"question": "What should I do if my order arrives damaged?"},
     "outputs": {"answer": "Contact support with photos of the damage so a replacement or refund can be processed; this is separate from the standard return window."}},
    {"inputs": {"question": "What are Bramblewood's support hours?"},
     "outputs": {"answer": "Support is available by chat and email, with a typical response time within one business day."}},
    {"inputs": {"question": "How often should tent rainflies be reproofed?"},
     "outputs": {"answer": "Once a year, with a DWR (durable water repellent) treatment."}},
    {"inputs": {"question": "What's the difference between the Summit Ridge and Alpine Trail tents?"},
     "outputs": {"answer": "The Summit Ridge 2P is a 3-season tent with a silicone-coated rainfly, while the Alpine Trail 4P is a 4-season expedition tent with a HydroShield-coated rainfly and reinforced poles."}},
    {"inputs": {"question": "Do aluminum tent poles crack?"},
     "outputs": {"answer": "They can develop hairline cracks near the ferrules after repeated setup in high wind; a cracked pole should be repaired with a splint, and a snapped pole is replaced under warranty."}},
]

for i, ex in enumerate(qa_examples):
    ex["metadata"] = {"source": "fieldwise_handbook", "row_index": i}

print("Support articles:", len(documents))
print("Evaluation examples:", len(qa_examples))
qa_examples[0]

## 4) Helpers for text overlap

Small, transparent text-overlap helpers used both by the local proxy
metrics later and by the evaluators.

In [ ]:
def normalize_text(text: str) -> str:
    return " ".join(str(text).lower().split())

def content_tokens(text: str) -> set:
    stop = {
        "the", "and", "that", "with", "from", "this", "what", "when", "where",
        "which", "were", "your", "you", "for", "are", "was", "how", "why", "who",
        "into", "have", "has", "had", "there", "their", "about", "does", "did",
    }
    toks = []
    for raw in normalize_text(text).split():
        token = "".join(ch for ch in raw if ch.isalnum())
        if len(token) > 3 and token not in stop:
            toks.append(token)
    return set(toks)

def overlap_ratio(a: str, b: str) -> float:
    a_tokens = content_tokens(a)
    b_tokens = content_tokens(b)
    if not a_tokens:
        return 0.0
    return len(a_tokens & b_tokens) / len(a_tokens)

## 5) Chunk and embed locally

We check once for a live embedding model and fall back to a deterministic
offline one if it isn't reachable, same as the rest of this module.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.embeddings import Embeddings
from langchain_chroma import Chroma
from sklearn.feature_extraction.text import HashingVectorizer
import shutil
from pathlib import Path

chunker = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=60)
chunks = chunker.split_documents(documents)

print("Chunks:", len(chunks))


class OfflineHashingEmbeddings(Embeddings):
    """Deterministic, dependency-free embedding fallback (see the
    end-to-end pipeline notebook for the full rationale)."""

    def __init__(self, n_features: int = 512):
        self.n_features = n_features
        self._hv = HashingVectorizer(
            n_features=n_features, alternate_sign=False, norm="l2", stop_words="english"
        )

    def embed_documents(self, texts):
        return self._hv.transform(texts).toarray().tolist()

    def embed_query(self, text):
        return self._hv.transform([text]).toarray()[0].tolist()


def get_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
        candidate = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
        candidate.embed_query("connectivity check")
        print("Embeddings: live sentence-transformers model loaded (semantic).")
        return candidate, "live"
    except Exception as e:
        print(f"Embeddings: falling back to offline hashing embeddings -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineHashingEmbeddings(), "offline"


embeddings, EMBEDDINGS_MODE = get_embeddings()

persist_dir = Path("./fieldwise_eval_data/chroma_store")
if persist_dir.exists():
    shutil.rmtree(persist_dir)
persist_dir.parent.mkdir(exist_ok=True)

vector_store = Chroma(
    collection_name="fieldwise_support_eval",
    embedding_function=embeddings,
    persist_directory=str(persist_dir),
)
vector_store.add_documents(chunks)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Chroma persistence directory:", persist_dir.resolve())

## 6) Create a LangSmith dataset

We check once whether the LangSmith API is reachable. If it is, we create
(or reuse) a dataset from the eval set above; if it isn't, we skip
straight to the local-only comparison later in this notebook.

In [ ]:
def check_langsmith():
    if not LANGSMITH_API_KEY:
        print("LangSmith: no API key set -- using local-only evaluation.")
        return None, "disabled"
    try:
        from langsmith import Client
        client = Client()
        client.has_dataset(dataset_name="__connectivity_check__")
        print("LangSmith: API reachable.")
        return client, "live"
    except Exception as e:
        print(f"LangSmith: API not reachable -- {type(e).__name__}: {str(e)[:160]}. Using local-only evaluation.")
        return None, "offline"


langsmith_client, LANGSMITH_MODE = check_langsmith()

dataset_name = "fieldwise-support-eval"

if LANGSMITH_MODE == "live":
    try:
        if not langsmith_client.has_dataset(dataset_name=dataset_name):
            dataset = langsmith_client.create_dataset(
                dataset_name=dataset_name,
                description="FieldWise support eval set built from Bramblewood's own knowledge base",
            )
            langsmith_client.create_examples(dataset_id=dataset.id, examples=qa_examples)
            print("Created dataset:", dataset_name)
        else:
            print("Dataset already exists:", dataset_name)
    except Exception as e:
        print(f"LangSmith dataset setup failed -- {type(e).__name__}: {str(e)[:160]}. Falling back to local-only evaluation.")
        LANGSMITH_MODE = "offline"

## 7) Groq availability check and the RAG chains

Two versions to compare:

- Baseline RAG: retrieve with the raw question and answer directly.
- Rewrite RAG: rewrite the question into a standalone search query first,
  then retrieve and answer.

We check once whether Groq is reachable and fall back to a small
deterministic mock chat model if not, so both chains run the same way
either way.

In [ ]:
import re
from typing import Any, List, Optional

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def _extract_section(text: str, label: str) -> str:
    """Pull a labeled section like 'Context:' out of a rendered prompt
    string, stopping at the next 'Label:' marker (or the end of the
    string) so it doesn't grab everything that follows."""
    pattern = rf"{re.escape(label)}:\s*(.*?)(?=\n\s*[A-Z][a-zA-Z ]{{0,20}}:|\Z)"
    match = re.search(pattern, text, flags=re.DOTALL)
    return match.group(1).strip() if match else ""


class OfflineMockChatModel(BaseChatModel):
    """Deterministic offline stand-in for ChatGroq. For a rewrite-style
    prompt it passes the question through unchanged (a safe no-op
    rewrite); for a context+question answer prompt it extracts the
    best-matching retrieved chunk and pulls 1-3 sentences from it. Every
    response is prefixed with "[offline mock]"."""

    @property
    def _llm_type(self) -> str:
        return "offline-mock-chat-model"

    def _generate(self, messages: List[BaseMessage], stop: Optional[List[str]] = None, run_manager: Optional[Any] = None, **kwargs: Any) -> ChatResult:
        system_text = " ".join(m.content for m in messages if m.type == "system" and isinstance(m.content, str))
        human_text = " ".join(m.content for m in messages if m.type == "human" and isinstance(m.content, str))

        if "standalone search query" in system_text.lower() or "rewrite" in system_text.lower():
            answer = human_text.strip()
        else:
            context = _extract_section(human_text, "Context") or human_text
            question = _extract_section(human_text, "Question") or human_text
            answer = self._extractive_answer(question, context)

        message = AIMessage(content=f"[offline mock] {answer}")
        return ChatResult(generations=[ChatGeneration(message=message)])

    @staticmethod
    def _extractive_answer(question: str, context: str, max_sentences: int = 2) -> str:
        q_terms = {w.lower() for w in re.findall(r"[a-zA-Z]{4,}", question)}

        chunk_blocks = [c.strip() for c in re.split(r"\n\s*\n", context.strip()) if c.strip()]
        if not chunk_blocks:
            chunk_blocks = [context.strip()]

        def term_overlap(text: str) -> int:
            text_terms = {w.lower() for w in re.findall(r"[a-zA-Z]{4,}", text)}
            return len(q_terms & text_terms)

        best_chunk = max(chunk_blocks, key=term_overlap)

        sentences = [
            s for s in re.split(r"(?<=[.!?])\s+", best_chunk.strip())
            if len(s.strip()) > 15
        ]
        scored = [(term_overlap(s), s.strip()) for s in sentences]
        scored.sort(key=lambda x: x[0], reverse=True)
        top = [s for _, s in scored[:max_sentences]] or (sentences[:1] if sentences else [""])
        return " ".join(top)[:600]


def get_llm():
    try:
        from langchain_groq import ChatGroq
        candidate = ChatGroq(model=GROQ_MODEL, temperature=0)
        candidate.invoke("connectivity check")
        print("LLM: live Groq model connected.")
        return candidate, "live"
    except Exception as e:
        print(f"LLM: falling back to offline mock chat model -- {type(e).__name__}: {str(e)[:160]}")
        return OfflineMockChatModel(), "offline"


llm, LLM_MODE = get_llm()

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer using only the provided context. If the context is insufficient, say you do not know."),
    ("user", "Question: {question}\n\nContext: {context}\n\nAnswer in 3 sentences or fewer."),
])

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", "Rewrite the question into a short standalone search query. Return only the rewritten query."),
    ("user", "{question}"),
])

rewrite_chain = rewrite_prompt | llm | StrOutputParser()

print("Groq/mock model ready. LLM mode:", LLM_MODE)

## 8) Utility: build context from retrieved chunks

In [ ]:
def make_context(docs, max_chars_per_doc=500):
    parts = []
    for d in docs:
        text = d.page_content.strip()
        if len(text) > max_chars_per_doc:
            text = text[:max_chars_per_doc].rsplit(" ", 1)[0] + "..."
        title = d.metadata.get("title", "unknown")
        parts.append(f"[{title}] {text}")
    return "\n\n".join(parts)

## 9) Baseline RAG target

Retrieves with the raw question and answers from the retrieved context.

In [ ]:
from langsmith import traceable

@traceable(name="rag_baseline")
def rag_baseline(inputs: dict) -> dict:
    question = inputs["question"]
    docs = retriever.invoke(question)
    context = make_context(docs)
    prompt_messages = answer_prompt.format_messages(question=question, context=context)
    response = llm.invoke(prompt_messages)

    return {
        "answer": response.content,
        "documents": [{"page_content": d.page_content, "metadata": d.metadata} for d in docs],
        "context": context,
    }

## 10) Rewrite RAG target

Rewrites the question into a standalone search query before retrieval,
which matters most when the incoming question is short or conversational.

In [ ]:
@traceable(name="rag_rewrite")
def rag_rewrite(inputs: dict) -> dict:
    question = inputs["question"]
    rewritten = rewrite_chain.invoke({"question": question})
    docs = retriever.invoke(rewritten)
    context = make_context(docs)
    prompt_messages = answer_prompt.format_messages(question=question, context=context)
    response = llm.invoke(prompt_messages)

    return {
        "answer": response.content,
        "documents": [{"page_content": d.page_content, "metadata": d.metadata} for d in docs],
        "context": context,
        "rewritten_question": rewritten,
    }

## 11) Local proxy metrics

Simple, transparent proxies -- not the full RAGAS library, but useful for a
fast first pass:
- faithfulness proxy: is the answer supported by the retrieved context?
- context precision proxy: is the retrieved context aligned with the
  question?
- recall proxy: is the reference answer supported by the retrieved
  context?

In [ ]:
def faithfulness_proxy(answer: str, context: str) -> float:
    return overlap_ratio(answer, context)

def context_precision_proxy(question: str, docs) -> float:
    if not docs:
        return 0.0
    question_tokens = content_tokens(question)
    if not question_tokens:
        return 0.0
    scores = []
    for d in docs:
        scores.append(len(question_tokens & content_tokens(d["page_content"])) / len(question_tokens))
    return sum(scores) / len(scores)

def recall_proxy(reference_answer: str, context: str) -> float:
    return overlap_ratio(reference_answer, context)

## 12) Define LangSmith evaluators

Same evaluation shape as LangSmith's own RAG tutorial (correctness,
groundedness, retrieval relevance), using the local proxy metrics above so
the notebook stays fast and easy to inspect.

In [ ]:
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    ref = normalize_text(reference_outputs.get("answer", ""))
    ans = normalize_text(outputs.get("answer", ""))
    if not ref or not ans:
        return False
    return overlap_ratio(ref, ans) >= 0.35

def groundedness(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    docs = outputs.get("documents", [])
    context = "\n".join(d.get("page_content", "") for d in docs)
    return faithfulness_proxy(outputs.get("answer", ""), context) >= 0.20

def retrieval_relevance(inputs: dict, outputs: dict, reference_outputs: dict = None) -> bool:
    docs = outputs.get("documents", [])
    return context_precision_proxy(inputs.get("question", ""), docs) >= 0.20

## 13) Run the LangSmith evaluation

`client.evaluate(target, data=dataset_name, evaluators=[...], experiment_prefix=...)`
is the main way to run an experiment in LangSmith. This only runs if the
dataset setup in Section 6 succeeded; otherwise we go straight to the
local-only comparison in Section 14, which always runs.

In [ ]:
baseline_experiment = None
rewrite_experiment = None

if LANGSMITH_MODE == "live":
    try:
        baseline_experiment = langsmith_client.evaluate(
            rag_baseline,
            data=dataset_name,
            evaluators=[correctness, groundedness, retrieval_relevance],
            experiment_prefix="baseline-fieldwise",
            metadata={"version": "baseline"},
            upload_results=True,
        )
        rewrite_experiment = langsmith_client.evaluate(
            rag_rewrite,
            data=dataset_name,
            evaluators=[correctness, groundedness, retrieval_relevance],
            experiment_prefix="rewrite-fieldwise",
            metadata={"version": "rewrite"},
            upload_results=True,
        )
        print("LangSmith evaluation launched.")
    except Exception as e:
        print(f"LangSmith evaluate() failed -- {type(e).__name__}: {str(e)[:160]}. Using the local-only comparison below.")
        LANGSMITH_MODE = "offline"
else:
    print("Skipping LangSmith evaluate() -- using the local-only comparison in Section 14 instead.")

## 14) Compare the two chain versions

LangSmith's comparison view is built for exactly this: compare experiments
side by side, spot regressions, and see which version does better on the
same dataset. This local comparison always runs, so you can see results
even without a live LangSmith connection.

In [ ]:
import pandas as pd

def experiment_to_df(experiment):
    if experiment is None:
        return pd.DataFrame()
    if hasattr(experiment, "to_pandas"):
        return experiment.to_pandas()
    try:
        return pd.DataFrame(list(experiment))
    except Exception:
        return pd.DataFrame()

baseline_df = experiment_to_df(baseline_experiment)
rewrite_df = experiment_to_df(rewrite_experiment)

print("LangSmith baseline results shape:", baseline_df.shape)
print("LangSmith rewrite results shape:", rewrite_df.shape)


def local_run(target_fn, examples, label):
    rows = []
    for ex in examples:
        inputs = ex["inputs"]
        reference = ex["outputs"]
        outputs = target_fn(inputs)
        docs = outputs.get("documents", [])
        context = outputs.get("context", "")
        rows.append({
            "experiment": label,
            "question": inputs["question"],
            "reference_answer": reference["answer"],
            "answer": outputs["answer"],
            "correctness_proxy": correctness(inputs, outputs, reference),
            "faithfulness_proxy": faithfulness_proxy(outputs["answer"], context),
            "context_precision_proxy": context_precision_proxy(inputs["question"], docs),
            "recall_proxy": recall_proxy(reference["answer"], context),
            "rewritten_question": outputs.get("rewritten_question", ""),
        })
    return rows

baseline_local = local_run(rag_baseline, qa_examples, "baseline")
rewrite_local = local_run(rag_rewrite, qa_examples, "rewrite")
local_compare = pd.DataFrame(baseline_local + rewrite_local)

local_compare.head()

## 15) A short summary table

A simple way to spot which version is doing better on this eval set.

In [ ]:
summary = (
    local_compare
    .groupby("experiment")[[
        "correctness_proxy",
        "faithfulness_proxy",
        "context_precision_proxy",
        "recall_proxy",
    ]]
    .mean()
    .reset_index()
)

summary

## 16) What to look at next

Useful next steps after this notebook:

- inspect failed examples in LangSmith (or in `local_compare` above)
- add better query rewriting for short, conversational tickets
- add contextual compression (see the `prompt_flows_in_rag` notebook)
- try a reranker (see the `naive_to_advanced_rag` notebook)
- expand the eval set as new ticket patterns show up
- replace the proxy scores with a stronger LLM-as-judge when ready

LangSmith supports dataset versioning, filtering, and comparing
experiments over time, which is what makes this an iterative loop rather
than a one-time check.

## Key takeaways

- This notebook evaluates FieldWise against its own support content, not a
  public benchmark -- which is what a real pre-launch check should do.
- Groq handles generation, Chroma stores local embeddings, and LangSmith
  manages the evaluation workflow when it's reachable.
- Two chain versions (baseline vs. rewrite) are compared instead of
  shipping on a single untested version.
- The local proxy metrics make faithfulness, context precision, and
  recall easy to inspect immediately, with or without LangSmith.
- Every external dependency here (embeddings, Groq, LangSmith) has a
  clearly labeled offline fallback, so the evaluation loop itself is
  always runnable, even before credentials are wired up.

## References

- Evaluation quickstart: https://docs.langchain.com/langsmith/evaluation-quickstart
- Evaluate a RAG application: https://docs.langchain.com/langsmith/evaluate-rag-tutorial
- Compare experiment results: https://docs.langchain.com/langsmith/compare-experiment-results
- Manage datasets programmatically: https://docs.langchain.com/langsmith/manage-datasets-programmatically
- Tracing quickstart: https://docs.langchain.com/langsmith/observability-quickstart
- Chroma local persistence: https://docs.langchain.com/oss/python/integrations/vectorstores/chroma

## Verification summary

| Section | Status | Notes |
|---|---|---|
| Package install | Executed | pip install cell |
| Knowledge base + eval set | Executed & verified | 10 articles, 16 QA pairs, hand-authored and grounded in the same content used elsewhere in this module |
| Embeddings + Chroma index | Executed & verified | see printed `EMBEDDINGS_MODE` |
| LangSmith dataset creation | Executed & verified | see printed `LANGSMITH_MODE` -- skipped cleanly when the API isn't reachable |
| Baseline / rewrite RAG chains | Executed & verified | see printed `LLM_MODE` -- offline mock in a sandbox with no Groq access, live Groq on a machine with `GROQ_API_KEY` set |
| LangSmith `evaluate()` | Executed only when `LANGSMITH_MODE == "live"` | needs a real LangSmith connection |
| Local proxy metrics + comparison table | Executed & verified | pure Python, always runs regardless of LangSmith/Groq availability |

On a machine with real `GROQ_API_KEY` and `LANGSMITH_API_KEY` values and
network access, `LLM_MODE` and `LANGSMITH_MODE` will both read `live`, and
the experiments will show up in the LangSmith UI for side-by-side
comparison, in addition to the local `summary` table above.